## Hyperparameter Tuning for Pose Training

## 1) Imports & Paths
Set up imports, project root detection, and output locations for grid-search logs/results.

In [4]:
from pathlib import Path
import itertools
import json
import os
import random
import re
import shlex
import subprocess
import sys
from datetime import datetime

import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "train").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not find project root from current notebook location.")

PROJECT_ROOT = find_project_root(Path.cwd())
TRAIN_SCRIPT = PROJECT_ROOT / "train" / "pose_train_predicted.py"
FEATURE_PATH = PROJECT_ROOT / "saved_models" / "stage3_predicted_keypoints.pt"
OUTPUT_DIR = PROJECT_ROOT / "training_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_CSV = OUTPUT_DIR / f"pose_grid_search_results_{RUN_STAMP}.csv"
RAW_LOG_DIR = OUTPUT_DIR / f"pose_grid_search_logs_{RUN_STAMP}"
RAW_LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Train script: {TRAIN_SCRIPT}")
print(f"Feature path: {FEATURE_PATH}")
print(f"Results CSV: {RESULTS_CSV}")
print(f"Log dir: {RAW_LOG_DIR}")

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(f"Missing training script: {TRAIN_SCRIPT}")
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Missing feature file: {FEATURE_PATH}")

Project root: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection
Train script: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/train/pose_train_predicted.py
Feature path: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/saved_models/stage3_predicted_keypoints.pt
Results CSV: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/training_outputs/pose_grid_search_results_20260310_024913.csv
Log dir: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/training_outputs/pose_grid_search_logs_20260310_024913


## 2) Define Grid
Choose hyperparameter values to search. Keep this compact initially, then expand once the pipeline works.

In [5]:
# Core fixed args
BASE_ARGS = {
    "feature_path": str(FEATURE_PATH),
    "epochs": 50,
    "batch_size": 64,
    "train_split": 0.7,
    "val_split": 0.15,
    "test_split": 0.15,
    "seed": 42,
    "early_stop_patience": 12,
}

# LARGE grid only
GRID = {
    "lr": [3e-4, 5e-4, 8e-4, 1e-3],
    "weight_decay": [1e-2, 7e-3, 5e-3, 1e-3],
    "hidden_dim": [256, 384, 512, 768],
    "dropout": [0.15, 0.2, 0.3, 0.4],
    "visibility_threshold": [0.0, 0.1, 0.15, 0.2],
    "label_smoothing": [0.0, 0.03, 0.05, 0.1],
    "xy_noise_std": [0.001, 0.003, 0.005, 0.01],
    "joint_dropout": [0.0, 0.02, 0.03, 0.05],
    "grad_clip_norm": [0.5, 1.0, 1.5, 2.0],
}

# Capped sampler controls
USE_CAPPED_SAMPLER = True
MAX_TRIALS = 120
SAMPLER_SEED = 42

def build_grid(grid_dict: dict) -> list[dict]:
    keys = list(grid_dict.keys())
    values = [grid_dict[k] for k in keys]
    combos = []
    for product_vals in itertools.product(*values):
        combos.append(dict(zip(keys, product_vals)))
    return combos

all_grid_combos = build_grid(GRID)
full_trial_count = len(all_grid_combos)

if USE_CAPPED_SAMPLER and MAX_TRIALS < full_trial_count:
    rng = random.Random(SAMPLER_SEED)
    sampled_indices = sorted(rng.sample(range(full_trial_count), k=MAX_TRIALS))
    grid_combos = [all_grid_combos[i] for i in sampled_indices]
else:
    sampled_indices = list(range(full_trial_count))
    grid_combos = all_grid_combos

print("Active preset: large")
print(f"Full trial space: {full_trial_count}")
print(f"Using capped sampler: {USE_CAPPED_SAMPLER}")
print(f"Selected trials: {len(grid_combos)}")
if USE_CAPPED_SAMPLER:
    print(f"Sampler seed: {SAMPLER_SEED}")
    print(f"Index span sample: first={sampled_indices[0]} last={sampled_indices[-1]}")

pd.DataFrame(grid_combos).head()

Active preset: large
Full trial space: 262144
Using capped sampler: True
Selected trials: 120
Sampler seed: 42
Index span sample: first=1701 last=261740


,lr,weight_decay,hidden_dim,dropout,visibility_threshold,label_smoothing,xy_noise_std,joint_dropout,grad_clip_norm
0,0.0003,0.01,256,0.20,0.15,0.05,0.005,0.02,1.0
1,0.0003,0.01,256,0.40,0.10,0.03,0.001,0.05,2.0
2,0.0003,0.01,384,0.20,0.20,0.05,0.001,0.00,2.0
3,0.0003,0.01,768,0.15,0.20,0.00,0.010,0.03,0.5
4,0.0003,0.01,768,0.20,0.15,0.03,0.003,0.03,0.5


## 3) Training Runner Helpers
Helper functions to build CLI args, execute each trial, parse metrics, and persist logs.

In [6]:
BEST_VAL_ACC_RE = re.compile(r"Best val acc:\s*([0-9]*\.?[0-9]+)")
TEST_ACC_RE = re.compile(r"Test loss:\s*([0-9]*\.?[0-9]+)\s+acc\s+([0-9]*\.?[0-9]+)")

def to_cli_args(args: dict) -> list[str]:
    cli = []
    for key, value in args.items():
        cli.append(f"--{key}")
        cli.append(str(value))
    return cli

def parse_metrics(stdout: str) -> dict:
    best_val_match = BEST_VAL_ACC_RE.search(stdout)
    test_match = TEST_ACC_RE.search(stdout)

    best_val_acc = float(best_val_match.group(1)) if best_val_match else None
    test_loss = float(test_match.group(1)) if test_match else None
    test_acc = float(test_match.group(2)) if test_match else None

    return {
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    }

def run_trial(trial_id: int, params: dict, dry_run: bool = False) -> dict:
    merged = {**BASE_ARGS, **params}
    cmd = [sys.executable, str(TRAIN_SCRIPT), *to_cli_args(merged)]
    cmd_str = " ".join(shlex.quote(part) for part in cmd)

    record = {
        "trial_id": trial_id,
        **params,
        "command": cmd_str,
        "status": "dry_run" if dry_run else "pending",
        "returncode": None,
        "best_val_acc": None,
        "test_loss": None,
        "test_acc": None,
    }

    if dry_run:
        return record

    run = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        text=True,
        capture_output=True,
    )

    record["returncode"] = int(run.returncode)
    record["status"] = "ok" if run.returncode == 0 else "failed"

    metrics = parse_metrics(run.stdout)
    record.update(metrics)

    log_path = RAW_LOG_DIR / f"trial_{trial_id:03d}.log"
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("# COMMAND\n")
        f.write(cmd_str + "\n\n")
        f.write("# STDOUT\n")
        f.write(run.stdout + "\n")
        f.write("\n# STDERR\n")
        f.write(run.stderr + "\n")

    record["log_path"] = str(log_path)
    return record

## 4) Execute Grid Search
Run all hyperparameter combinations. Set `DRY_RUN=True` first to verify commands.

In [7]:
DRY_RUN = False
STOP_ON_FAILURE = False

all_records = []
for trial_id, params in enumerate(grid_combos, start=1):
    print(f"[{trial_id}/{len(grid_combos)}] Running: {params}")
    record = run_trial(trial_id=trial_id, params=params, dry_run=DRY_RUN)
    record["sampled_from_full_index"] = sampled_indices[trial_id - 1] if trial_id - 1 < len(sampled_indices) else None
    all_records.append(record)

    status = record["status"]
    val = record.get("best_val_acc")
    test = record.get("test_acc")
    print(f"  -> status={status}, best_val_acc={val}, test_acc={test}")

    if (not DRY_RUN) and STOP_ON_FAILURE and status == "failed":
        print("Stopping early due to failure.")
        break

results_df = pd.DataFrame(all_records)
if not results_df.empty:
    results_df.to_csv(RESULTS_CSV, index=False)

print(f"\nTrials logged: {len(results_df)}")
print(f"Results saved to: {RESULTS_CSV}")
results_df.head()

[1/120] Running: {'lr': 0.0003, 'weight_decay': 0.01, 'hidden_dim': 256, 'dropout': 0.2, 'visibility_threshold': 0.15, 'label_smoothing': 0.05, 'xy_noise_std': 0.005, 'joint_dropout': 0.02, 'grad_clip_norm': 1.0}
  -> status=ok, best_val_acc=0.8066666666666666, test_acc=0.73
[2/120] Running: {'lr': 0.0003, 'weight_decay': 0.01, 'hidden_dim': 256, 'dropout': 0.4, 'visibility_threshold': 0.1, 'label_smoothing': 0.03, 'xy_noise_std': 0.001, 'joint_dropout': 0.05, 'grad_clip_norm': 2.0}
  -> status=ok, best_val_acc=0.7666666666666667, test_acc=0.7433
[3/120] Running: {'lr': 0.0003, 'weight_decay': 0.01, 'hidden_dim': 384, 'dropout': 0.2, 'visibility_threshold': 0.2, 'label_smoothing': 0.05, 'xy_noise_std': 0.001, 'joint_dropout': 0.0, 'grad_clip_norm': 2.0}
  -> status=ok, best_val_acc=0.8366666666666667, test_acc=0.7467
[4/120] Running: {'lr': 0.0003, 'weight_decay': 0.01, 'hidden_dim': 768, 'dropout': 0.15, 'visibility_threshold': 0.2, 'label_smoothing': 0.0, 'xy_noise_std': 0.01, 'joint

,trial_id,lr,weight_decay,hidden_dim,dropout,visibility_threshold,label_smoothing,xy_noise_std,joint_dropout,grad_clip_norm,command,status,returncode,best_val_acc,test_loss,test_acc,log_path,sampled_from_full_index
0,1,0.0003,0.01,256,0.20,0.15,0.05,0.005,0.02,1.0,/Users/hanyiliu/anaconda3/bin/python /Users/ha...,ok,0,0.806667,5.1596,0.7300,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,1701
1,2,0.0003,0.01,256,0.40,0.10,0.03,0.001,0.05,2.0,/Users/hanyiliu/anaconda3/bin/python /Users/ha...,ok,0,0.766667,0.6878,0.7433,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,3407
2,3,0.0003,0.01,384,0.20,0.20,0.05,0.001,0.00,2.0,/Users/hanyiliu/anaconda3/bin/python /Users/ha...,ok,0,0.836667,0.5931,0.7467,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,6019
3,4,0.0003,0.01,768,0.15,0.20,0.00,0.010,0.03,0.5,/Users/hanyiliu/anaconda3/bin/python /Users/ha...,ok,0,0.820000,21.6339,0.7733,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,13112
4,5,0.0003,0.01,768,0.20,0.15,0.03,0.003,0.03,0.5,/Users/hanyiliu/anaconda3/bin/python /Users/ha...,ok,0,0.803333,25.2919,0.7433,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,13912


## 5) Analyze Results
Sort by validation/test metrics, inspect failures, and export the best trial config.

In [10]:
if results_df.empty:
    print("No results recorded yet.")
else:
    display_cols = [
        "trial_id", "status", "best_val_acc", "test_acc", "test_loss",
        "lr", "weight_decay", "hidden_dim", "dropout", "visibility_threshold",
        "label_smoothing", "xy_noise_std", "joint_dropout", "grad_clip_norm",
    ]
    display_cols = [c for c in display_cols if c in results_df.columns]

    ranked = results_df.sort_values(
        by=["test_acc", "best_val_acc"],
        ascending=[False, False],
        na_position="last",
    )

    print("Top 10 trials:")
    display(ranked[display_cols].head(10))

    failed = results_df[results_df["status"] == "failed"] if "status" in results_df.columns else pd.DataFrame()
    print(f"Failed trials: {len(failed)}")
    if len(failed) > 0:
        display(failed[[c for c in ["trial_id", "returncode", "log_path", "command"] if c in failed.columns]])

    best_ok = ranked[ranked["status"] == "ok"] if "status" in ranked.columns else ranked
    if len(best_ok) > 0:
        best_row = best_ok.iloc[0].to_dict()
        best_params = {k: best_row[k] for k in GRID.keys() if k in best_row}
        best_config_path = OUTPUT_DIR / f"pose_grid_search_best_config_{RUN_STAMP}.json"
        with open(best_config_path, "w", encoding="utf-8") as f:
            json.dump(best_params, f, indent=2)
        print(f"Best config saved to: {best_config_path}")
        print("Best trial summary:")
        print({
            "trial_id": best_row.get("trial_id"),
            "best_val_acc": best_row.get("best_val_acc"),
            "test_acc": best_row.get("test_acc"),
            "test_loss": best_row.get("test_loss"),
        })

Top 10 trials:


,trial_id,status,best_val_acc,test_acc,test_loss,lr,weight_decay,hidden_dim,dropout,visibility_threshold,label_smoothing,xy_noise_std,joint_dropout,grad_clip_norm
61,62,ok,0.823333,0.7833,0.6101,0.0005,0.001,384,0.40,0.00,0.10,0.001,0.00,1.0
86,87,ok,0.816667,0.7800,199.2558,0.0008,0.005,768,0.30,0.00,0.05,0.005,0.05,1.0
118,119,ok,0.830000,0.7767,0.6010,0.0010,0.001,768,0.15,0.15,0.05,0.010,0.03,0.5
37,38,ok,0.823333,0.7767,0.6345,0.0005,0.010,768,0.30,0.10,0.00,0.001,0.02,0.5
89,90,ok,0.820000,0.7767,0.6347,0.0008,0.001,384,0.20,0.20,0.05,0.003,0.03,0.5
28,29,ok,0.833333,0.7733,12338.2456,0.0003,0.001,512,0.20,0.00,0.00,0.001,0.00,1.0
87,88,ok,0.830000,0.7733,298.6325,0.0008,0.005,768,0.30,0.00,0.10,0.003,0.02,1.0
3,4,ok,0.820000,0.7733,21.6339,0.0003,0.010,768,0.15,0.20,0.00,0.010,0.03,0.5
109,110,ok,0.806667,0.7733,0.6366,0.0010,0.007,512,0.40,0.10,0.03,0.005,0.05,1.5
108,109,ok,0.846667,0.7700,0.6677,0.0010,0.007,512,0.15,0.10,0.05,0.001,0.00,2.0


Failed trials: 0
Best config saved to: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/training_outputs/pose_grid_search_best_config_20260310_024913.json
Best trial summary:
{'trial_id': 62, 'best_val_acc': 0.8233333333333334, 'test_acc': 0.7833, 'test_loss': 0.6101}


## 6) Optional: Run Best Config Again
After selecting the best config, optionally run one confirmation training pass.

In [12]:
RUN_CONFIRMATION = True

if not RUN_CONFIRMATION:
    print("Skipping confirmation run (set RUN_CONFIRMATION=True to execute).")
else:
    if "best_row" not in locals():
        raise RuntimeError("Run the analysis cell first to compute best_row.")

    confirm_params = {k: best_row[k] for k in GRID.keys() if k in best_row}
    confirm_record = run_trial(
        trial_id=999,
        params=confirm_params,
        dry_run=False,
    )
    print("Confirmation run result:")
    print(confirm_record)

Confirmation run result:
{'trial_id': 999, 'lr': 0.0005, 'weight_decay': 0.001, 'hidden_dim': 384, 'dropout': 0.4, 'visibility_threshold': 0.0, 'label_smoothing': 0.1, 'xy_noise_std': 0.001, 'joint_dropout': 0.0, 'grad_clip_norm': 1.0, 'command': '/Users/hanyiliu/anaconda3/bin/python /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/train/pose_train_predicted.py --feature_path /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/saved_models/stage3_predicted_keypoints.pt --epochs 50 --batch_size 64 --train_split 0.7 --val_split 0.15 --test_split 0.15 --seed 42 --early_stop_patience 12 --lr 0.0005 --weight_decay 0.001 --hidden_dim 384 --dropout 0.4 --visibility_threshold 0.0 --label_smoothing 0.1 --xy_noise_std 0.001 --joint_dropout 0.0 --grad_clip_norm 1.0', 'status': 'ok', 'returncode': 0, 'best_val_acc': 0.8233333333333334, 'test_loss': 0.6101, 'test_acc': 0.7833, 'log_path': '/Users/hanyiliu/Documents/GitHub/tennis-pose-detection/training_outputs/pose_grid_search_logs_2026031

In [13]:
print(confirm_record["command"])

/Users/hanyiliu/anaconda3/bin/python /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/train/pose_train_predicted.py --feature_path /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/saved_models/stage3_predicted_keypoints.pt --epochs 50 --batch_size 64 --train_split 0.7 --val_split 0.15 --test_split 0.15 --seed 42 --early_stop_patience 12 --lr 0.0005 --weight_decay 0.001 --hidden_dim 384 --dropout 0.4 --visibility_threshold 0.0 --label_smoothing 0.1 --xy_noise_std 0.001 --joint_dropout 0.0 --grad_clip_norm 1.0
